# Fine-Tune Qwen2.5-3B for Arabic Teaching (SageMaker)

**Model:** Qwen2.5-3B-Instruct with LoRA (using Unsloth)

**Training data:** ~115 conversations (teaching, grading, exercise generation)

**Method:** LoRA (rank=32, alpha=32)

**Max sequence length:** 1536 tokens

**Output:** 4-bit quantized model (~2GB)

---

## Setup for SageMaker

1. **Instance Type:** ml.g4dn.xlarge or ml.g5.xlarge (T4/A10G GPU)
2. **HF Token:** Set as environment variable `HF_TOKEN` or in cell below
3. **Training data:** Upload `combined_training_data.jsonl` to EFS at `/home/sagemaker-user/user-default-efs/arabic-teaching/data/`
4. **Run all cells** (~20-30 minutes)
5. **Model output:** Saved to EFS at `/home/sagemaker-user/user-default-efs/arabic-teaching/models/qwen-3b-arabic-teaching/`

## Cell 1: Install Dependencies

In [1]:
%%capture
# Simple installation - use SageMaker's default PyTorch
!pip install --upgrade torchvision
!pip install unsloth trl transformers datasets accelerate bitsandbytes
!pip install -U peft
!pip uninstall -y pyarrow
!pip install pyarrow --no-cache-dir

## ⚠️ RESTART KERNEL - Then run all cells below

## Cell 2: HuggingFace Auth

In [2]:
# Check what's using space
!du -sh /home/sagemaker-user/* 2>/dev/null | sort -h

# Clear everything possible from default storage
!rm -rf ~/.cache/*
!rm -rf /home/sagemaker-user/.cache/*


import os

from huggingface_hub import login

EFS_ROOT = "/home/sagemaker-user/user-default-efs"
HF_CACHE = os.path.join(EFS_ROOT, "hf_cache")

# 1. Create the directories on the BIG drive
os.makedirs(HF_CACHE, exist_ok=True)

# 2. Redirect ALL possible cache/temp variables
os.environ["HF_HOME"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = os.path.join(EFS_ROOT, "hf_cache")
os.environ["HF_DATASETS_CACHE"] = os.path.join(EFS_ROOT, "hf_cache")

# Now check space
!df -h /home/sagemaker-user

print(f"✅ All traffic redirected to EFS: {HF_CACHE}")
login(token="<HF_TOKEN>")

0	/home/sagemaker-user/user-default-efs
Filesystem      Size  Used Avail Use% Mounted on
/dev/nvme2n1     10G  109M  9.9G   2% /home/sagemaker-user
✅ All traffic redirected to EFS: /home/sagemaker-user/user-default-efs/hf_cache


## Cell 3: Imports & Check GPU

In [3]:
import gc
import json
import os

import torch

# IMPORTANT: Disable Unsloth statistics BEFORE importing Unsloth
os.environ["UNSLOTH_DISABLE_LOG_STATS"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from datasets import Dataset
from trl import SFTTrainer
from unsloth import FastLanguageModel

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
if torch.cuda.is_available():
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected! Training will be very slow.")

/opt/conda/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-04-17 04:01:04.484963: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776398464.511654     300 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776398464.522252     300 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-17 04:01:04.727846: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SS

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch: 2.10.0+cu128
GPU: NVIDIA L4
Memory: 23.7 GB


## Cell 4: Configuration

In [ ]:
# EFS paths (adjust if your EFS mount is different)
EFS_ROOT = "/home/sagemaker-user/user-default-efs"
DATA_DIR = f"{EFS_ROOT}/data"

# Combined output
COMBINED_DATA = f"{DATA_DIR}/agent1_teaching_training_data_v2.jsonl"

print("📦 Combining training data files...\n")
DATA_PATH = f"{COMBINED_DATA}"
OUTPUT_DIR = f"{EFS_ROOT}/models/qwen-3b-arabic-teaching"


# Use the non-Unsloth version (Unsloth will patch it automatically)
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

# LoRA config
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.1
LORA_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training config
BATCH_SIZE = 2
GRAD_ACCUM = 4
EPOCHS = 6
LR = 2e-4
WARMUP = 0.03

print("=" * 60)
print("🚀 Fine-Tuning Qwen2.5-3B for Arabic Teaching")
print("=" * 60)
print(f"Model: {MODEL_NAME}")
print(f"Data: {DATA_PATH}")
print(f"Output: {OUTPUT_DIR}")
print(f"Batch: {BATCH_SIZE}, Accum: {GRAD_ACCUM}, Epochs: {EPOCHS}")
print(f"Max Seq Length: {MAX_SEQ_LENGTH}")
print("=" * 60)

# Verify data file exists
if not os.path.exists(DATA_PATH):
    print(f"\n❌ Training data not found at: {DATA_PATH}")
    print("\nPlease upload combined_training_data.jsonl to EFS at:")
    print(f"   {os.path.dirname(DATA_PATH)}/")
else:
    print("\n✅ Training data found")

# Verify EFS cache is setup
print(f"\n📁 HuggingFace cache: {os.environ.get('HF_HOME', 'NOT SET')}")
if os.path.exists(os.environ.get("HF_HOME", "")):
    print("✅ EFS cache directory exists")
else:
    print("❌ EFS cache directory NOT found - run Cell 2 first!")

📦 Combining training data files...

🚀 Fine-Tuning Qwen2.5-3B for Arabic Teaching
Model: Qwen/Qwen2.5-7B-Instruct
Data: /home/sagemaker-user/user-default-efs/data/agent1_teaching_training_data_v2.jsonl
Output: /home/sagemaker-user/user-default-efs/models/qwen-3b-arabic-teaching
Batch: 2, Accum: 4, Epochs: 6
Max Seq Length: 2048

✅ Training data found

📁 HuggingFace cache: /home/sagemaker-user/user-default-efs/hf_cache
✅ EFS cache directory exists


## Cell 5: Load Training Data

In [5]:
conversations = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            conversations.append(json.loads(line))

print(f"✅ Loaded {len(conversations)} conversations")
print(f"   Steps/epoch: {len(conversations) // (BATCH_SIZE * GRAD_ACCUM)}")
print(f"   Total steps: {(len(conversations) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS}")

✅ Loaded 153 conversations
   Steps/epoch: 19
   Total steps: 114


## Cell 6: Load Model with Unsloth

In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
    token="<HF_TOKEN>",
)
print(f"✅ Model loaded: {model.num_parameters() / 1e9:.2f}B params")

==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.16G [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Model loaded: 7.62B params


## Cell 7: Add LoRA Adapters

## Cell 8: Format Dataset

In [7]:
formatted_data = [
    {
        "text": tokenizer.apply_chat_template(
            c["messages"], tokenize=False, add_generation_prompt=False
        )
    }
    for c in conversations
]

dataset = Dataset.from_list(formatted_data)
print(f"✅ Dataset: {len(dataset)} examples")

✅ Dataset: 153 examples


In [8]:
# Validation split
eval_size = 20  # Use 20 examples for validation
train_dataset = dataset.select(range(len(dataset) - eval_size))
eval_dataset = dataset.select(range(len(dataset) - eval_size, len(dataset)))

print(f"✅ Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

✅ Train: 133, Eval: 20


## Cell 9: Setup Trainer

In [9]:
import torch
from peft import LoraConfig
from trl import SFTConfig

# 1. Clear memory
torch.cuda.empty_cache()
gc.collect()

# 2. Define the PEFT configuration (matches your LORA settings)
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model.add_adapter(peft_config, adapter_name="my_adapter")

# 3. Define the SFTConfig (v1.0.0 requirement)
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    eval_strategy="epoch",  # ADD THIS
    save_strategy="epoch",  # ADD THIS
    load_best_model_at_end=True,  # ADD THIS
    metric_for_best_model="loss",  # ADD THIS
    logging_steps=1,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",
    average_tokens_across_devices=False,
)

# 4. Initialize Trainer exactly as shown in the second docs
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=sft_config,
    processing_class=tokenizer,  # <--- Replaces 'tokenizer'
)

trainer.model.set_adapter("my_adapter")

print("✅ Trainer ready! The PEFT config is now explicitly linked.")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/133 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/20 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
✅ Trainer ready! The PEFT config is now explicitly linked.


## Cell 10: Train (~20-30 min on g4dn.xlarge)

In [10]:
import logging

# Disable the specific logger that's crashing on the closed Jupyter stream
logging.getLogger("transformers.trainer").setLevel(logging.ERROR)

print("🚀 Training starting...")

try:
    trainer.train()
except ValueError as e:
    if "I/O operation on closed file" in str(e):
        print("Caught Jupyter logging error, but training is proceeding in the background.")
        # Training often continues even if the progress bar display fails
    else:
        raise e

print("\n✅ Check your EFS 'models' folder for checkpoints!")

🚀 Training starting...
Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,1.193200,1.233848
2,0.047900,0.188991
3,0.014600,0.070439
4,0.011800,0.033658
5,0.012700,0.019348
6,0.015500,0.017925


FileNotFoundError: [Errno 2] No such file or directory: '/home/sagemaker-user/user-default-efs/models/qwen-3b-arabic-teaching/checkpoint-102/pytorch_model.bin'

## Cell 11: Save Model to EFS

In [11]:
# Cell 11: Save Model to EFS
import os
import tarfile

# Save to EFS (persists across notebook sessions)
lora_output_dir = f"{OUTPUT_DIR}/lora_adapters"
model.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)

print(f"✅ Saved to EFS: {lora_output_dir}")
print("\n📁 Model persisted on EFS at:")
print(f"   {lora_output_dir}")

print("\n📦 Creating downloadable archive...")

# Create tarball in /tmp (accessible for download)
archive_path = f"{OUTPUT_DIR}/qwen-3b-arabic-teaching-lora.tar.gz"
with tarfile.open(archive_path, "w:gz") as tar:
    tar.add(lora_output_dir, arcname="lora_adapters")

archive_size = os.path.getsize(archive_path) / (1024 * 1024)
print(f"✅ Archive created: {archive_path} ({archive_size:.1f} MB)")

print("\n📥 To download:")
print("   1. File browser → /tmp/qwen-3b-arabic-teaching-lora.tar.gz")
print("   2. Right-click → Download")
print("   3. Extract on your machine")

print("\nModel files in archive:")
for file in os.listdir(lora_output_dir):
    file_path = os.path.join(lora_output_dir, file)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"   - {file} ({size_mb:.1f} MB)")

✅ Saved to EFS: /home/sagemaker-user/user-default-efs/models/qwen-3b-arabic-teaching/lora_adapters

📁 Model persisted on EFS at:
   /home/sagemaker-user/user-default-efs/models/qwen-3b-arabic-teaching/lora_adapters

📦 Creating downloadable archive...
✅ Archive created: /home/sagemaker-user/user-default-efs/models/qwen-3b-arabic-teaching/qwen-3b-arabic-teaching-lora.tar.gz (33.9 MB)

📥 To download:
   1. File browser → /tmp/qwen-3b-arabic-teaching-lora.tar.gz
   2. Right-click → Download
   3. Extract on your machine

Model files in archive:
   - tokenizer_config.json (0.0 MB)
   - adapter_model.safetensors (38.5 MB)
   - adapter_config.json (0.0 MB)
   - merges.txt (1.6 MB)
   - generation_config.json (0.0 MB)
   - special_tokens_map.json (0.0 MB)
   - chat_template.jinja (0.0 MB)
   - added_tokens.json (0.0 MB)
   - tokenizer.json (10.9 MB)
   - vocab.json (2.6 MB)


## Cell 12: Test Generation

In [33]:
import json
import os
import sys

!pip install python-Levenshtein

# 1. Setup Paths
# We point directly to the 'src' directory so that 'agents' is a top-level package
eval_src_path = os.path.expanduser("/home/sagemaker-user/user-default-efs/evaluation/src")

if eval_src_path not in sys.path:
    sys.path.insert(0, eval_src_path)

# 2. Import the Agent
# Since we added 'src' to the path, we import directly from agents
try:
    from agents.teaching_agent import TeachingAgent

    print("✓ Successfully imported TeachingAgent")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("Ensure __init__.py exists in /evaluation/src/agents/")

# 3. Initialize Agent
print("Initializing TeachingAgent...")
teaching_agent = TeachingAgent(
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
)
print("✓ TeachingAgent initialized\n")

# 4. Load test scenarios
# Note: eval_dir is used here just to locate the data file
eval_dir = os.path.expanduser("/home/sagemaker-user/user-default-efs/evaluation")
E2E_TEST_PATH = os.path.join(eval_dir, "data/evaluation/teaching_agent/e2e_conversation_flows.json")

print(f"Loading E2E test scenarios from {E2E_TEST_PATH}...")
with open(E2E_TEST_PATH, encoding="utf-8") as f:
    test_data = json.load(f)
    scenarios = test_data["scenarios"]

print(f"✓ Loaded {len(scenarios)} conversation scenarios\n")

✓ Successfully imported TeachingAgent
Initializing TeachingAgent...
✓ TeachingAgent initialized

Loading E2E test scenarios from /home/sagemaker-user/user-default-efs/evaluation/data/evaluation/teaching_agent/e2e_conversation_flows.json...
✓ Loaded 12 conversation scenarios



In [34]:
%%time
import time

print("=" * 80)
print("RUNNING E2E CONVERSATION EVALUATION")
print("=" * 80)
print()

all_results = []
start_time = time.time()

for i, scenario in enumerate(scenarios, 1):
    scenario_start = time.time()

    print(f"\n{'=' * 80}")
    print(f"Scenario {i}/{len(scenarios)}: {scenario['test_id']}")
    print(f"Description: {scenario['description']}")
    print(f"Turns: {len(scenario['turns'])}")
    print(f"{'=' * 80}\n")

    result = {
        "test_id": scenario["test_id"],
        "description": scenario["description"],
        "category": scenario.get("category", "unknown"),
        "total_turns": len(scenario["turns"]),
        "turn_results": [],
        "overall_pass": True,
        "issues": [],
    }

    for turn_data in scenario["turns"]:
        turn_num = turn_data["turn"]
        mode = turn_data["mode"]
        input_data = turn_data["input"]
        expected_behaviors = turn_data.get("expected_behaviors", [])

        # Extract user response if present
        user_response = turn_data.get(
            "user_response", turn_data.get("user_message", input_data.get("user_message", ""))
        )

        print(f"Turn {turn_num}: {mode}")
        print(
            f"  Expected: {', '.join(expected_behaviors[:3])}{'...' if len(expected_behaviors) > 3 else ''}"
        )

        # Get agent response
        try:
            if mode == "lesson_start":
                response = teaching_agent.handle_lesson_start(input_data)
            elif mode == "teaching_vocab":
                response = teaching_agent.handle_teaching_vocab(input_data)
            elif mode == "teaching_grammar":
                response = teaching_agent.handle_teaching_grammar(input_data)
            elif mode == "feedback_vocab":
                response = teaching_agent.handle_feedback_vocab(input_data)
            elif mode == "feedback_grammar":
                response = teaching_agent.handle_feedback_grammar(input_data)
            else:
                # General handler for edge cases
                response = teaching_agent.handle_input(
                    user_input=input_data.get("user_message", ""), student_context=input_data
                )
        except Exception as e:
            print(f"  ❌ Error generating response: {e}")
            response = f"[Error: {str(e)}]"
            result["overall_pass"] = False
            result["issues"].append(f"Turn {turn_num}: Generation error")

        # Log response preview
        preview = response
        print(f"  Response: {preview}\n")

        result["turn_results"].append(
            {
                "turn": turn_num,
                "mode": mode,
                "user_response": user_response,
                "response": response,
                "expected_behaviors": expected_behaviors,
            }
        )

    all_results.append(result)

    scenario_time = time.time() - scenario_start
    print(f"✓ Scenario {i} complete ({scenario_time:.1f}s)\n")

total_time = time.time() - start_time

print("\n" + "=" * 80)
print("EVALUATION COMPLETE")
print("=" * 80)
print(f"Total scenarios: {len(all_results)}")
print(f"Total time: {total_time / 60:.1f} minutes ({total_time:.1f}s)")
print(f"Average per scenario: {total_time / len(all_results):.1f}s\n")

RUNNING E2E CONVERSATION EVALUATION


Scenario 1/12: e2e_vocab_learning_happy_path
Description: Complete vocab learning flow: lesson start → batch intro → quiz with mixed results
Turns: 4

Turn 1: lesson_start
  Expected: greets_warmly, presents_vocabulary_count, presents_grammar_topics...
  Response: Welcome to Lesson 1! 🌟 I'm so excited to start this Arabic learning journey with you!

Today you'll learn:
- **Vocabulary:** 10 new words (كِتَاب/book, قَلَم/pen, etc.)
- **Grammar:** 2 topics (Feminine Nouns, Definite Article)

Where would you like to start?
1. Start with vocabulary (10 words)
2. Start with grammar (2 topics)
3. See lesson progress

Or tell me what you'd like to do!

Turn 2: teaching_vocab
  Expected: shows_3_words_with_arabic, mentions_flashcards, offers_quiz_next_all_options
  Response: Let's learn Batch 1! Here are your first 3 words:

📚 كِتَاب (kitaab) - book
🡴 قَلَم (qalam) - pen
_Office_ 桌子 (maktab) - desk

Take your time learning with the flashcards!

What would y

In [35]:
from datetime import datetime

# Prepare results
output = {
    "metadata": {
        "model": "Qwen2.5-7B Teaching (LoRA V2) - SageMaker GPU",
        "evaluation_date": datetime.now().isoformat(),
        "device": "ml.g4dn.xlarge (T4 GPU)",
        "test_type": "End-to-end conversation flows",
    },
    "summary": {
        "total_scenarios": len(all_results),
        "total_time_seconds": total_time,
        "scenarios": all_results,
    },
}

# Save results
output_path = "/home/sagemaker-user/user-default-efs/e2e_results_sagemaker.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"✓ Results saved to {output_path}")
print(f"\nEvaluated {len(all_results)} scenarios successfully!")
print("\n📥 Download this file to analyze results locally")

✓ Results saved to /home/sagemaker-user/user-default-efs/e2e_results_sagemaker.json

Evaluated 12 scenarios successfully!

📥 Download this file to analyze results locally


In [36]:
# Quick summary of results
print("=" * 80)
print("QUICK RESULTS SUMMARY")
print("=" * 80)
print()

# Count by category
categories = {}
for result in all_results:
    cat = result["category"]
    if cat not in categories:
        categories[cat] = {"total": 0, "passed": 0}
    categories[cat]["total"] += 1
    if result["overall_pass"]:
        categories[cat]["passed"] += 1

for cat, data in sorted(categories.items()):
    pass_rate = data["passed"] / data["total"] if data["total"] > 0 else 0
    status = "✅" if pass_rate >= 0.8 else "⚠️" if pass_rate >= 0.5 else "❌"
    print(f"{status} {cat}: {data['passed']}/{data['total']} ({pass_rate:.1%})")

# Overall
total_passed = sum(1 for r in all_results if r["overall_pass"])
overall_rate = total_passed / len(all_results) if all_results else 0
print(f"\n{'=' * 80}")
print(f"Overall: {total_passed}/{len(all_results)} ({overall_rate:.1%})")
print(f"{'=' * 80}\n")

# Check for issues
print("\nIssues Found:")
issue_count = 0
for result in all_results:
    if result["issues"]:
        print(f"\n{result['test_id']}:")
        for issue in result["issues"]:
            print(f"  - {issue}")
            issue_count += 1

if issue_count == 0:
    print("  None! 🎉")
else:
    print(f"\nTotal issues: {issue_count}")

QUICK RESULTS SUMMARY

✅ edge_case: 4/4 (100.0%)
✅ error_recovery: 2/2 (100.0%)
✅ happy_path: 2/2 (100.0%)
✅ navigation: 4/4 (100.0%)

Overall: 12/12 (100.0%)


Issues Found:
  None! 🎉


In [ ]:
# Show first scenario's responses
print("=" * 80)
print("SAMPLE RESPONSES - First Scenario")
print("=" * 80)
print()

first_scenario = all_results[0]
print(f"Scenario: {first_scenario['test_id']}")
print(f"Description: {first_scenario['description']}\n")

for turn in first_scenario["turn_results"][:3]:  # First 3 turns
    print(f"\nTurn {turn['turn']} ({turn['mode']}):")
    print("-" * 80)
    print(turn["response"])
    print("-" * 80)